In [1]:
import pandas as pd
import numpy as np
import json
import re
from ast import literal_eval
import importlib
from dotenv import load_dotenv
import os
os.chdir('..')


In [2]:
# Load environment variables from .env file
load_dotenv()

True

In [3]:
# Set GEMINI_API_KEY environment variable with your API key
os.environ['GEMINI_API_KEY'] = os.getenv('GEMINI_API_KEY')

In [4]:
import base64
from google import genai
from google.genai import types
from copy import deepcopy

In [5]:
def generate(prompt):
	client = genai.Client(
			api_key=os.environ.get("GEMINI_API_KEY"),
		)

	model = "gemini-2.5-flash"
	contents = [
		types.Content(
			role="user",
			parts=[
				types.Part.from_text(
					text=prompt
				),
			],
		),
	]
	generate_content_config = types.GenerateContentConfig(
		temperature=0.75,
		top_p=0.9,
		top_k=40,
		max_output_tokens=8192,
		thinking_config=types.ThinkingConfig(thinking_budget=-1), # Dynamic thinking = -1, no thinking = 0
		response_mime_type="application/json",
		system_instruction=[
			types.Part.from_text(
				text="""You are an expert in Natural Language Processing. You are also linguist with an expertise in Indonesian and English."""
			),
		],
	)
		
	response = client.models.generate_content(
		model=model, contents=contents, config=generate_content_config
	)
	# print(response.text)
	return response

from typing import List, Dict
import re
def parse_absa_string(text: str) -> List[Dict[str, str]]:
    """
    Parses a string formatted as "[A] aspect [O] opinion [S] sentiment" into a list of dictionaries.
    Each dictionary contains the tag as the key and the corresponding value.
    For example, "[A] [O] [S] [A] harga [O] terjangkau [S] positive [SSEP] [A] fasilitas [O] nyaman [S] positive" becomes:
    [{'A': 'harga', 'S': 'positive', 'O': 'terjangkau'},
    {'A': 'fasilitas', 'S': 'positive', 'O': 'nyaman'}].

    Args:
        text (str): ABSA string output to be parsed.

    Returns:
        List[Dict[str, str]]: List of dictionaries of parsed ABSA output.

    """
    pattern = r"\[(\w+)\]\s*([^[]+)"
    matches = re.findall(pattern, text)

    result = []
    current_dict = {}

    for tag, content in matches:
        if tag == "SSEP":  # Sentence separator -> Start a new dictionary
            result.append(current_dict)
            current_dict = {}
        else:
            current_dict[tag] = content.strip()

    if current_dict:  # Append the last sentence if it exists
        result.append(current_dict)

    return result

In [14]:
dataset_folder = 'mvp_aos'
dataset_type = 'hotel_reviews'
lang = 'indo'
split = 'train'
dataset_path = f'dataset/{dataset_type}/{lang}/{dataset_folder}/{split}.json'
with open(dataset_path, 'r') as f:
	dataset = json.load(f)


In [15]:
dataset_unique = []
for instance in dataset:
	if instance['element_order'] == 'aos':
		dataset_unique.append(deepcopy(instance))

In [16]:
dataset_single_triplet = []
for instance in dataset_unique:
	triplets = parse_absa_string(instance['target'])
	for triplet in triplets:
		new_instance = deepcopy(instance)
		new_instance['target'] = f"[A] {triplet.get('A', '')} [O] {triplet.get('O', '')} [S] {triplet.get('S', '')}"
		dataset_single_triplet.append(new_instance)

In [17]:
df_dataset = pd.DataFrame(dataset_unique)
df_dataset

,sentence_id,instance_id,input,target,element_order,task_elements
0,0,0,kamar saya ada kendala di ac tidak berfungsi o...,[A] ac [O] tidak berfungsi optimal [S] negativ...,aos,aos
1,1,5,tempatnya bagus . kolam renangnya bersih . [A]...,[A] tempatnya [O] bagus [S] positive\n[A] kola...,aos,aos
2,2,10,"oke banget , tetapi ac nya tidak bisa diatur s...",[A] ac nya [O] tidak bisa diatur suhu nya [S] ...,aos,aos
3,3,15,keren . nyaman semuanya . [A] [O] [S],[A] semuanya [O] nyaman [S] positive\n[A] null...,aos,aos
4,4,20,"tidak dapat snack . setelah di keluhan , baru ...",[A] snack [O] tidak dapat [S] negative,aos,aos
...,...,...,...,...,...,...
2477,2495,12475,wifi kurang joss . [A] [O] [S],[A] wifi [O] kurang joss [S] negative,aos,aos
2478,2496,12480,"kamar cukup bersih , hanya sempit , . [A] [O] [S]",[A] kamar [O] cukup bersih [S] positive\n[A] k...,aos,aos
2479,2497,12485,"nyaman , bersih , dan pelayananya sangat ramah...",[A] pelayananya [O] sangat ramah [S] positive\...,aos,aos
2480,2498,12490,sangat kecewa dengan kamar dan pelayanan stafn...,[A] kamar [O] sangat kecewa [S] negative\n[A] ...,aos,aos


In [18]:
check_list = df_dataset['sentence_id'].to_list()
# Check if check_list is ordered from smallest to largest (missing indexes is allowed, just make sure the order is correct)
is_ordered = all(earlier <= later for earlier, later in zip(check_list, check_list[1:]))
print(f"Is the sentence_id list ordered? {is_ordered}")

Is the sentence_id list ordered? True


In [20]:
with open(f'notebooks/prompt_translate/translate-triplet-jav.txt', 'r') as f:
	prompt_template = f.read()
print(prompt_template)

### Instruction
You will be given input-output pairs of Aspect Sentiment Triplet Extraction.
Given an Indonesian text and triplets consist of aspect term, opinion term, and sentiment polarity, translate all of them to Ngoko-level of Javanese. Ngoko-level Javanese is the informal variety of Javanese used in daily conversations between friends or people of the same age.
The order of the triplet is (aspect term, opinion term, sentiment polarity).
Below is the definition of each element in the triplet:
- The aspect term refers to a specific feature, attribute, or aspect of a product or service on which a user can express an opinion. Explicit aspect terms appear explicitly as a substring of the given text. The aspect term might be “NULL” for the implicit aspect.
- The sentiment polarity refers to the degree of positivity, negativity or neutrality expressed in the opinion towards a particular aspect or feature of a product or service, and the available polarities include: “positive”, “negati

### Individual API request

In [36]:
from tqdm import tqdm
from time import sleep
from ast import literal_eval

outputs = {}
outputs_text = {}
for idx, row in tqdm(df_dataset.iterrows(), desc="Translating triplets", total=df_dataset.shape[0]):
	input_text = row['input'].replace('[A] [O] [S]', '').strip()
	triplets = parse_absa_string(row['target'])
	target_text = []
	for triplet in triplets:
		target_text.append(f"({triplet.get('A', 'err_empty')}, {triplet.get('O', 'err_empty')}, {triplet.get('S', 'err_empty')})")
	target_text = '[' + ', '.join(target_text) + ']'
	prompt = prompt_template.replace('{text-input}', input_text).replace('{input-triplets}', target_text)
	while True:
		try:
			output = generate(prompt)
			outputs[idx] = output
			outputs_text[idx] = literal_eval(output.text)
			outputs_text[idx]['text'] = input_text
			outputs_text[idx]['triplets'] = target_text
			break
		except Exception as e:
			print(f"Error: {e}")
			sleep(3.0)  # Wait for 3 seconds before retrying
			continue
	
	# Write to json file after each successful generation
	os.makedirs(f'temp/translation_output/eng/{dataset_folder}', exist_ok=True)
	with open(f'temp/translation_output/eng/{dataset_folder}/{os.path.basename(dataset_path)}', 'w') as f:
		json.dump(outputs_text, f, indent=4, ensure_ascii=False)
	
	if idx == 10:
		break

Translating triplets:   0%|          | 10/2482 [00:59<4:06:51,  5.99s/it]


### Batch API request

In [21]:
inline_requests = []
temp_for_jsonl = []
for idx, row in df_dataset.iterrows():
	input_text = row['input'].replace('[A] [O] [S]', '').strip()
	triplets = parse_absa_string(row['target'])
	target_text = []
	for triplet in triplets:
		target_text.append(f"({triplet.get('A', 'err_empty')}, {triplet.get('O', 'err_empty')}, {triplet.get('S', 'err_empty')})")
	target_text = '[' + ', '.join(target_text) + ']'
	prompt = prompt_template.replace('{text-input}', input_text).replace('{input-triplets}', target_text)
	
	inline_request = {
		"contents": [
			{
				"role": "user",
				"parts": [
					{
						"text": prompt
					}
				]
			}
		],
		"generation_config": {
			"temperature": 0.75,
			"top_p": 0.9,
			"top_k": 40,
			"max_output_tokens": 8192,
			"thinking_config": {
				"thinking_budget": -1
			},
			"response_mime_type": "application/json",
		},
		"system_instruction": {
			"parts": [
				{
					'text': "You are an expert in Natural Language Processing. You are also linguist with an expertise in Indonesian and Javanese."
				}
			]
		}
	}
	inline_requests.append(inline_request)

	temp_for_jsonl.append({
		'key': f'request-{row["sentence_id"]}',
		'request': inline_request
	})
	# if idx == 4:
	# 	break

In [22]:
len(inline_requests), len(temp_for_jsonl)

(2482, 2482)

In [23]:
# print(inline_requests[-10]['contents'][0]['parts'][0]['text'])
print(temp_for_jsonl[-1])

{'key': 'request-2499', 'request': {'contents': [{'role': 'user', 'parts': [{'text': '### Instruction\nYou will be given input-output pairs of Aspect Sentiment Triplet Extraction.\nGiven an Indonesian text and triplets consist of aspect term, opinion term, and sentiment polarity, translate all of them to Ngoko-level of Javanese. Ngoko-level Javanese is the informal variety of Javanese used in daily conversations between friends or people of the same age.\nThe order of the triplet is (aspect term, opinion term, sentiment polarity).\nBelow is the definition of each element in the triplet:\n- The aspect term refers to a specific feature, attribute, or aspect of a product or service on which a user can express an opinion. Explicit aspect terms appear explicitly as a substring of the given text. The aspect term might be “NULL” for the implicit aspect.\n- The sentiment polarity refers to the degree of positivity, negativity or neutrality expressed in the opinion towards a particular aspect o

In [24]:
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

In [25]:
lang_target = 'jav'

In [26]:
# Create a jsonl file for inline requests
os.makedirs(f'temp/translation_output/{lang_target}/{dataset_folder}', exist_ok=True)
with open(f'temp/translation_output/{lang_target}/{dataset_folder}/inline_requests_{lang_target}_{split}.jsonl', 'w') as f:
	for item in temp_for_jsonl:
		f.write(json.dumps(item, ensure_ascii=False) + '\n')

In [27]:
# Upload the file to the File API
uploaded_file = client.files.upload(
    file=f'temp/translation_output/{lang_target}/{dataset_folder}/inline_requests_{lang_target}_{split}.jsonl',
    config=types.UploadFileConfig(display_name=f'inline_requests_{lang_target}_{split}', mime_type='jsonl')
)

print(f"Uploaded file: {uploaded_file.name}")

Uploaded file: files/2r5jaouu9wdj


In [29]:
from google import genai

# Assumes `uploaded_file` is the file object from the previous step
file_batch_job = client.batches.create(
    model="gemini-2.5-flash",
    src=uploaded_file.name,
    config={
        'display_name': f"file-upload-job-translation-{lang_target}-{split}-fixed",
    },
)

print(f"Created batch job: {file_batch_job.name}")

Created batch job: batches/lpxh4xaknvxckykmrk9hjbazv87anyyklal4


In [30]:
# List all batches
batches = client.batches.list()
for batch in batches:
    print(f"Batch ID: {batch.name}, State: {batch.state.name}, Display Name: {batch.display_name}")

Batch ID: batches/lpxh4xaknvxckykmrk9hjbazv87anyyklal4, State: JOB_STATE_PENDING, Display Name: file-upload-job-translation-jav-train-fixed
Batch ID: batches/9udm06rnexogkfmqcxmav7oblt1g6lm467z3, State: JOB_STATE_SUCCEEDED, Display Name: file-upload-job-translation-jav-test
Batch ID: batches/jbu1m7rk7953auz9a0xipwxxl2bh14i0q42j, State: JOB_STATE_SUCCEEDED, Display Name: file-upload-job-translation-jav-dev
Batch ID: batches/675kvbrqpt3igvutlowozqsdjjxlalad5eaf, State: JOB_STATE_SUCCEEDED, Display Name: file-upload-job-translation-jav-train
Batch ID: batches/65azlnc2oetwg4x6np88dz9wgmwsyo98ki9o, State: JOB_STATE_SUCCEEDED, Display Name: file-upload-job-translation-sunda-dev-all
Batch ID: batches/mucz1j6fvlq0n4krwws4528xk181u4lxfnqy, State: JOB_STATE_SUCCEEDED, Display Name: file-upload-job-translation-eng-dev-all
Batch ID: batches/jyjjxzt3feea2hrx8ft25cpwu7e1mdxhge3r, State: JOB_STATE_SUCCEEDED, Display Name: file-upload-job-translation-sunda-train-all
Batch ID: batches/j64igjih81kj18rzk

In [31]:
from time import sleep

# Use the name of the job you want to check
# e.g., inline_batch_job.name from the previous step
job_name = "batches/lpxh4xaknvxckykmrk9hjbazv87anyyklal4"  # (e.g. 'batches/your-batch-id')
batch_job = client.batches.get(name=job_name)

completed_states = set([
    'JOB_STATE_SUCCEEDED',
    'JOB_STATE_FAILED',
    'JOB_STATE_CANCELLED',
    'JOB_STATE_EXPIRED',
])

print(f"Polling status for job: {job_name}")
batch_job = client.batches.get(name=job_name) # Initial get
while batch_job.state.name not in completed_states:
  print(f"Current state: {batch_job.state.name}")
  sleep(30) # Wait for 30 seconds before polling again
  batch_job = client.batches.get(name=job_name)

print(f"Job finished with state: {batch_job.state.name}")
if batch_job.state.name == 'JOB_STATE_FAILED':
    print(f"Error: {batch_job.error}")

Polling status for job: batches/lpxh4xaknvxckykmrk9hjbazv87anyyklal4
Current state: JOB_STATE_PENDING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Job finished with state: JOB_STATE_SUCCEEDED


In [25]:
from tqdm import tqdm

In [26]:
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

# Use the name of the job you want to check
# e.g., inline_batch_job.name from the previous step
job_name = "batches/65azlnc2oetwg4x6np88dz9wgmwsyo98ki9o"
batch_job = client.batches.get(name=job_name)

outputs = {}
outputs_text = {}

if batch_job.state.name == 'JOB_STATE_SUCCEEDED':

	# If batch job was created with a file
	if batch_job.dest and batch_job.dest.file_name:
		# Results are in a file
		result_file_name = batch_job.dest.file_name
		print(f"Results are in file: {result_file_name}")

		print("Downloading result file content...")
		file_content = client.files.download(file=result_file_name)
		# Process file_content (bytes) as needed
		lines = file_content.decode('utf-8').splitlines()
		for i, line in tqdm(enumerate(lines)):
			result = json.loads(line)
			key = result['key'].split('-')[-1]

			# print(result)
			if 'response' in result:
				try:
					outputs_text[key] = literal_eval(result['response']['candidates'][0]['content']['parts'][0]['text'])
				except Exception as e:
					print(f"Error in Response {i+1}: {e}")
					outputs_text[key] = result['response']  # Fallback
					outputs_text[key] = {
						'translated_text': '',
						'translated_triplets': [],
					}
			elif 'error' in result:
				print(f"Error: {result['error']}")

	# If batch job was created with inline request
	# (for embeddings, use batch_job.dest.inlined_embed_content_responses)
	elif batch_job.dest and batch_job.dest.inlined_responses:
		# Results are inline
		print("Results are inline:")
		for i, inline_response in tqdm(enumerate(batch_job.dest.inlined_responses)):
			if inline_response.response:
				# Accessing response, structure may vary.
				try:
					outputs_text[i] = literal_eval(inline_response.response.text)
				except Exception as e:
					print(f"Error in Response {i+1}: {e}")
					outputs_text[i] = inline_response.response  # Fallback
					outputs_text[i] = {
						'translated_text': '',
						'translated_triplets': [],
					}
			elif inline_response.error:
				print(f"Error: {inline_response.error}")
	else:
		print("No results found (neither file nor inline).")
else:
	print(f"Job did not succeed. Final state: {batch_job.state.name}")
	if batch_job.error:
		print(f"Error: {batch_job.error}")

Results are in file: files/batch-65azlnc2oetwg4x6np88dz9wgmwsyo98ki9o


1000it [00:00, 17495.15it/s]


In [27]:
# for i, inline_response in tqdm(enumerate(batch_job.dest.inlined_responses)):
# 	if inline_response.response:
# 		# Accessing response, structure may vary.
# 		try:
# 			print(f"----------------- Response {i+1} -----------------")
# 			print(literal_eval(inline_response.response.text))
# 			print(inline_requests[i]['contents'][0]['parts'][0]['text'].split('### Inference:')[-1].strip())
# 		except Exception as e:
# 			print(f"Error in Response {i+1}: {e}")
# 			# outputs[i] = inline_response.response  # Fallback
# 			print(inline_response.response.text)
# 	elif inline_response.error:
# 		print(f"Error: {inline_response.error}")

In [28]:
dataset_path

'hotel_dataset/indo/corrected_splitopinion_typocorrected_aos/hotel_aste_dev_augmented.json'

In [29]:
save_path = f'temp/translation_output/{lang_target}/{dataset_folder}/{os.path.basename(dataset_path)}'

In [30]:
save_path

'temp/translation_output/sunda/corrected_splitopinion_typocorrected_aos/hotel_aste_dev_augmented.json'

In [31]:
os.makedirs(f'temp/translation_output/{lang_target}/{dataset_folder}', exist_ok=True)
with open(save_path, 'w') as f:
	json.dump(outputs_text, f, indent=4, ensure_ascii=False)
print(f"Saved translated outputs to {save_path}")

Saved translated outputs to temp/translation_output/sunda/corrected_splitopinion_typocorrected_aos/hotel_aste_dev_augmented.json


## Preprocess

In [32]:
with open(save_path, 'r') as f:
	outputs_text = json.load(f)

In [33]:
def add_space_around_punctuation(text):
    # Except for '-'
    # Ensure space before punctuation
    text = re.sub(r'(\S)([.,!?\(\)\"\';:+/]+)', r'\1 \2', text)
    # Ensure space after punctuation
    text = re.sub(r'([.,!?\(\)\"\';:+/]+)(\S)', r'\1 \2', text)
    # Replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)
    # Ensure punctuation sequences like '...' are split into spaced dots
    text = re.sub(r'([.]{2,})', lambda m: ' '.join(m.group(1)), text)
    return text.strip()

In [34]:
len(outputs_text)

1000

In [35]:
df_dataset

,sentence_id,instance_id,task_elements,input,target,element_order
0,2500,12500,aos,pintu geser kurang rapat . [A] [O] [S],[A] pintu geser [O] kurang rapat [S] negative,aos
1,2501,12505,aos,pelayanan lumayan baik . [A] [O] [S],[A] pelayanan [O] lumayan baik [S] positive,aos
2,2502,12510,aos,air bersih untuk mck tidak ada . [A] [O] [S],[A] air bersih [O] tidak ada [S] negative,aos
3,2503,12515,aos,ternyata ada makanan ringan gratis . [A] [O] [S],[A] makanan ringan gratis [O] ada [S] positive,aos
4,2504,12520,aos,wifi buruk suka down . [A] [O] [S],[A] wifi [O] buruk [S] negative,aos
...,...,...,...,...,...,...
995,3495,17475,aos,"strategis , dekat jalan raya . [A] [O] [S]",[A] null [O] strategis [S] positive [SSEP] [A]...,aos
996,3496,17480,aos,( - ) tempatnya kurang bersih . tipetipe pengi...,[A] tempatnya [O] kurang bersih [S] negative [...,aos
997,3497,17485,aos,"mantapp , pelayanan prima . [A] [O] [S]",[A] null [O] mantapp [S] positive [SSEP] [A] p...,aos
998,3498,17490,aos,"hotel lumayan nyaman sih , tetapi dikamar saya...",[A] hotel [O] lumayan nyaman [S] positive [SSE...,aos


In [36]:
def check_mismatches_triplet_format(outputs_text):
	mismatch_indexes = []
	mismatch_notes = {}
	for key, instance in outputs_text.items():
		translated_text = add_space_around_punctuation(instance['translated_text'].lower()).strip()
		mismatched = False
		for triplet in instance['translated_triplets']:
			aspect_term = add_space_around_punctuation(triplet['aspect_term'].lower()).strip()
			opinion_term = add_space_around_punctuation(triplet['opinion_term'].lower()).strip()
			if aspect_term not in translated_text and aspect_term != 'null':
				print(f"Mismatch in instance {key}: aspect_term '{aspect_term}' not found in {translated_text}")
				mismatch_notes[key] = mismatch_notes.get(key, []) + [f"aspect_term '{aspect_term}' not found"]
				mismatched = True
			if opinion_term not in translated_text and opinion_term != 'null':
				print(f"Mismatch in instance {key}: opinion_term '{opinion_term}' not found in {translated_text}")
				mismatch_notes[key] = mismatch_notes.get(key, []) + [f"opinion_term '{opinion_term}' not found"]
				mismatched = True
		if mismatched:
			mismatch_indexes.append(key)
	print(f"Total mismatches found: {len(mismatch_indexes)}")
	return mismatch_indexes, mismatch_notes
mismatch_indexes, mismatch_notes = check_mismatches_triplet_format(outputs_text)

Mismatch in instance 2524: opinion_term 'err_empty' not found in harga kahontal jeung deukeut ka pusat kota bandung .
Mismatch in instance 2572: opinion_term 'angger wé kitu malah beuki parah' not found in kualitas wifina goréng pisan , malah kadang-kadang sok paéh sababaraha kali . urang ngéndong di dieu , tapi kualitas wifina angger wé kitu , malah beuki parah . tulung diperhatikeun .
Mismatch in instance 2632: opinion_term 'aya di lamping pasir' not found in lokasi nu alami , ayana di lamping pasir . matak nimbulkeun suasana alam nu kacida éndahna .
Mismatch in instance 2646: opinion_term 'deukeut ti supermarket' not found in ac-na teu tiis jeung panto kamar mandi bocor , tapi sacara umum mah alus , utamana dahareunana . deukeut ti tempat dahar di luar hotél jeung supermarket .
Mismatch in instance 2688: opinion_term 'loba cucunguk' not found in loba reungit jeung cucunguk . lanténa ogé kotor pisan !
Mismatch in instance 2702: opinion_term 'tulungan dioméan' not found in tempat runt

In [37]:
mismatch_indexes = list(set(mismatch_indexes))
len(mismatch_indexes)

22

In [38]:
input_eng = {}
for key, instance in outputs_text.items():
	translated_text = add_space_around_punctuation(instance['translated_text'].lower()).strip()
	input_eng[key] = f'{translated_text} [A] [O] [S]'

target_eng = {}
for key, instance in outputs_text.items():
	triplet_texts = []
	for triplet in instance['translated_triplets']:
		aspect_term = add_space_around_punctuation(triplet['aspect_term'].lower()).strip()
		opinion_term = add_space_around_punctuation(triplet['opinion_term'].lower()).strip()
		sentiment = triplet['sentiment_polarity'].lower().strip()
		triplet_texts.append(f"[A] {aspect_term} [O] {opinion_term} [S] {sentiment}")
	target_eng[key] = ' [SSEP] '.join(triplet_texts)
print(len(input_eng), len(target_eng))

1000 1000


In [39]:
# Sort input_eng and target_eng by key to match df_dataset order
input_eng = dict(sorted(input_eng.items(), key=lambda item: int(item[0])))
target_eng = dict(sorted(target_eng.items(), key=lambda item: int(item[0])))

In [40]:
list(input_eng.values())

['panto geser kurang rapet . [A] [O] [S]',
 'palayananana lumayan alus . [A] [O] [S]',
 'cai bersih keur mck euweuh . [A] [O] [S]',
 'geuning aya cemilan haratis . [A] [O] [S]',
 'wifi goréng , mindeng down . [A] [O] [S]',
 'punten , perhatikeun deui kalengkepan kamarna . [A] [O] [S]',
 'suasana na kurang ngeunah . pikeun ngéndong kudu deposit 300 rébu , tapi bisa diganti ku ktp . [A] [O] [S]',
 'kamar-kamar nu hawana lega di dieu téh pangnyaman-nyamanna . [A] [O] [S]',
 'cai panasna téh palsu . [A] [O] [S]',
 'harga murah . loba tempat wisata peuting di sabudeureunana . [A] [O] [S]',
 'butuh pangharum kamar mandi ? nu séjénna mah mantap . [A] [O] [S]',
 'ngan ari mawa mobil mah , parkirna hésé . [A] [O] [S]',
 'dahareun nu kurang nyugemakeun . [A] [O] [S]',
 'kabersihanana tingkatkeun deui . [A] [O] [S]',
 'pituduh lokasi kudu dioméan . [A] [O] [S]',
 'pas rék check out , pelayananana euweuh pisan , jadi kapaksa ngadagoan lila . [A] [O] [S]',
 'tempatna ngeunah keur istirahat . [A] [O

In [41]:
# df_dataset = df_dataset.loc[df_dataset['sentence_id'] >= 1006, :].copy()

In [42]:
# df_dataset.reset_index(drop=True, inplace=True)

In [43]:
df_dataset

,sentence_id,instance_id,task_elements,input,target,element_order
0,2500,12500,aos,pintu geser kurang rapat . [A] [O] [S],[A] pintu geser [O] kurang rapat [S] negative,aos
1,2501,12505,aos,pelayanan lumayan baik . [A] [O] [S],[A] pelayanan [O] lumayan baik [S] positive,aos
2,2502,12510,aos,air bersih untuk mck tidak ada . [A] [O] [S],[A] air bersih [O] tidak ada [S] negative,aos
3,2503,12515,aos,ternyata ada makanan ringan gratis . [A] [O] [S],[A] makanan ringan gratis [O] ada [S] positive,aos
4,2504,12520,aos,wifi buruk suka down . [A] [O] [S],[A] wifi [O] buruk [S] negative,aos
...,...,...,...,...,...,...
995,3495,17475,aos,"strategis , dekat jalan raya . [A] [O] [S]",[A] null [O] strategis [S] positive [SSEP] [A]...,aos
996,3496,17480,aos,( - ) tempatnya kurang bersih . tipetipe pengi...,[A] tempatnya [O] kurang bersih [S] negative [...,aos
997,3497,17485,aos,"mantapp , pelayanan prima . [A] [O] [S]",[A] null [O] mantapp [S] positive [SSEP] [A] p...,aos
998,3498,17490,aos,"hotel lumayan nyaman sih , tetapi dikamar saya...",[A] hotel [O] lumayan nyaman [S] positive [SSE...,aos


In [44]:
mismatch_indexes

['2709',
 '3366',
 '3028',
 '3155',
 '3210',
 '3476',
 '3032',
 '2646',
 '2524',
 '2702',
 '2688',
 '3246',
 '3190',
 '3496',
 '2778',
 '3117',
 '2881',
 '2572',
 '2632',
 '2790',
 '2932',
 '2838']

In [45]:
index_check = df_dataset['sentence_id'].to_list()
df_dataset[f'input_{lang_target}'] = list(input_eng.values())
df_dataset[f'target_{lang_target}'] = list(target_eng.values())
df_dataset['aspect_or_opinion_not_in_input'] = [str(idx) in mismatch_indexes for idx in index_check]
df_dataset['mismatch_notes'] = [mismatch_notes.get(str(idx), []) for idx in index_check]
df_dataset['mismatch_notes_format'] = df_dataset['mismatch_notes'].apply(lambda x: '\n'.join(x))

In [46]:
df_dataset['target_format'] = df_dataset['target'].apply(lambda x: '\n'.join(x.split(' [SSEP] ')))
df_dataset[f'target_format_{lang_target}'] = df_dataset[f'target_{lang_target}'].apply(lambda x: '\n'.join(x.split(' [SSEP] ')))

In [47]:
df_dataset

,sentence_id,instance_id,task_elements,input,target,element_order,input_sunda,target_sunda,aspect_or_opinion_not_in_input,mismatch_notes,mismatch_notes_format,target_format,target_format_sunda
0,2500,12500,aos,pintu geser kurang rapat . [A] [O] [S],[A] pintu geser [O] kurang rapat [S] negative,aos,panto geser kurang rapet . [A] [O] [S],[A] panto geser [O] kurang rapet [S] negative,False,[],,[A] pintu geser [O] kurang rapat [S] negative,[A] panto geser [O] kurang rapet [S] negative
1,2501,12505,aos,pelayanan lumayan baik . [A] [O] [S],[A] pelayanan [O] lumayan baik [S] positive,aos,palayananana lumayan alus . [A] [O] [S],[A] palayanan [O] lumayan alus [S] positive,False,[],,[A] pelayanan [O] lumayan baik [S] positive,[A] palayanan [O] lumayan alus [S] positive
2,2502,12510,aos,air bersih untuk mck tidak ada . [A] [O] [S],[A] air bersih [O] tidak ada [S] negative,aos,cai bersih keur mck euweuh . [A] [O] [S],[A] cai bersih [O] euweuh [S] negative,False,[],,[A] air bersih [O] tidak ada [S] negative,[A] cai bersih [O] euweuh [S] negative
3,2503,12515,aos,ternyata ada makanan ringan gratis . [A] [O] [S],[A] makanan ringan gratis [O] ada [S] positive,aos,geuning aya cemilan haratis . [A] [O] [S],[A] cemilan haratis [O] aya [S] positive,False,[],,[A] makanan ringan gratis [O] ada [S] positive,[A] cemilan haratis [O] aya [S] positive
4,2504,12520,aos,wifi buruk suka down . [A] [O] [S],[A] wifi [O] buruk [S] negative,aos,"wifi goréng , mindeng down . [A] [O] [S]",[A] wifi [O] goréng [S] negative,False,[],,[A] wifi [O] buruk [S] negative,[A] wifi [O] goréng [S] negative
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,3495,17475,aos,"strategis , dekat jalan raya . [A] [O] [S]",[A] null [O] strategis [S] positive [SSEP] [A]...,aos,"stratégis , deukeut jalan raya . [A] [O] [S]",[A] null [O] stratégis [S] positive [SSEP] [A]...,False,[],,[A] null [O] strategis [S] positive\n[A] null ...,[A] null [O] stratégis [S] positive\n[A] null ...
996,3496,17480,aos,( - ) tempatnya kurang bersih . tipetipe pengi...,[A] tempatnya [O] kurang bersih [S] negative [...,aos,( - ) tempatna kurang beresih . tipe-tipe pang...,[A] tempatna [O] kurang beresih [S] negative [...,True,[aspect_term 'penginepan' not found],aspect_term 'penginepan' not found,[A] tempatnya [O] kurang bersih [S] negative\n...,[A] tempatna [O] kurang beresih [S] negative\n...
997,3497,17485,aos,"mantapp , pelayanan prima . [A] [O] [S]",[A] null [O] mantapp [S] positive [SSEP] [A] p...,aos,"mantep pisan , palayananana gé alus pisan . [A...",[A] null [O] mantep pisan [S] positive [SSEP] ...,False,[],,[A] null [O] mantapp [S] positive\n[A] pelayan...,[A] null [O] mantep pisan [S] positive\n[A] pa...
998,3498,17490,aos,"hotel lumayan nyaman sih , tetapi dikamar saya...",[A] hotel [O] lumayan nyaman [S] positive [SSE...,aos,"hotél lumayan genah mah , tapi di kamar kuring...",[A] hotél [O] lumayan genah [S] positive [SSEP...,False,[],,[A] hotel [O] lumayan nyaman [S] positive\n[A]...,[A] hotél [O] lumayan genah [S] positive\n[A] ...


In [48]:
df_dataset[['sentence_id', 'input', f'input_{lang_target}', 'target_format', f'target_format_{lang_target}', 'aspect_or_opinion_not_in_input', 'mismatch_notes_format']].rename({'target_format': 'target', f'target_format_{lang_target}': f'target_{lang_target}', 'mismatch_notes_format': 'mismatch_notes'}).to_csv(f'temp/translation_output/{lang_target}/{dataset_folder}/translated_dataset_{lang}_{split}.csv', index=False)

### Check mismatches after annotation

In [33]:
def get_google_sheet(sheet_id: str, sheet_gid: str) -> pd.DataFrame:
	"""
	Downloads a specific sheet from a Google Sheet into a pandas DataFrame.

	Args:
		sheet_id: The ID of the Google Sheet.
		sheet_gid: The GID of the specific sheet to download.

	Returns:
		A pandas DataFrame containing the data from the specified sheet.
	"""
	url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={sheet_gid}'
	df = pd.read_csv(url)
	return df
google_sheet_id = '1_ZEFErp75wDNkXIvYflxv1d4nAQZrMlg_Vin4cgpyyY'  # Replace with your actual Google Sheet ID
gid_eng_train = '1363651964'  # Replace with the actual GID for the English sheet
gid_eng_test = '1183495517'
gid_sunda_test = '1775717233'
split = 'test'
lang_target = 'sunda'
try:
	df_translated_correction = get_google_sheet(google_sheet_id, gid_sunda_test)
	print("Successfully loaded data from the specific sheet:")
except Exception as e:
	print(f"An error occurred: {e}")
	print("Please ensure the Google Sheet is shared correctly and the IDs are correct.")

Successfully loaded data from the specific sheet:


In [34]:
df_translated_correction

,sentence_id,input,input_sunda,target_format,input_sunda_corrected,target_format_sunda,target_format_sunda_corrected,aspect_or_opinion_not_in_input,mismatch_notes_format
0,3500,pelayanan nya sangat ramah . [A] [O] [S],layananna someah pisan . [A] [O] [S],[A] pelayanan nya [O] sangat ramah [S] positive,palayananna someah pisan . [A] [O] [S],[A] layananna [O] someah pisan [S] positive,[A] palayananna [O] someah pisan [S] positive,FALSE,NaN
1,3501,sayang wifi tidak bagus harus keluar kamar . [...,"hanjakal wifi téh teu alus , kudu kaluar kamar...",[A] wifi [O] tidak bagus harus keluar kamar [S...,NaN,"[A] wifi [O] teu alus , kudu kaluar kamar [S] ...",NaN,FALSE,NaN
2,3502,"tulisannya twin bed , tetapi yang ada kamarnya...","tulisanana twin bed , tapi nu aya mah kamarna ...",[A] kamarnya [O] beda [S] negative,"tulisanna mah dua ranjang , ngan nu aya kamarn...",[A] kamarna [O] béda [S] negative,NaN,FALSE,NaN
3,3503,"over all baik , hanya sja akan lebih memuaskan...","sakabéhna mah alus , ngan wé bakal leuwih nyug...",[A] over all [O] baik [S] positive\n[A] air ho...,"sakabéhna mah alus , ngan bakal leuwih nyugema...",[A] sakabéhna [O] alus [S] positive\n[A] cai p...,NaN,FALSE,NaN
4,3504,fasilatas sesuia . [A] [O] [S],fasilitasna cocog . [A] [O] [S],[A] fasilitas [O] sesuai [S] positive,fasilitasna luyu . [A] [O] [S],[A] fasilitas [O] cocog [S] positive,[A] fasilitasna [O] luyu [S] positive,FALSE,NaN
...,...,...,...,...,...,...,...,...,...
995,4495,"lumayan , harga murah banged . [A] [O] [S]","lumayan , harga mirah pisan . [A] [O] [S]",[A] harga [O] murah banget [S] positive\n[A] n...,NaN,[A] harga [O] mirah pisan [S] positive\n[A] nu...,NaN,FALSE,NaN
996,4496,buat lakilaki dan perempuan yang belum menikah...,pikeun lalaki jeung awéwé anu can nikah ogé me...,[A] null [O] buat laki laki dan perempuan yang...,keur lalaki jeung awéwé anu can nikah ogé meun...,[A] null [O] pikeun lalaki jeung awéwé anu can...,[A] null [O] keur lalaki jeung awéwé anu can n...,FALSE,NaN
997,4497,"kamar sangat nyaman dan bersih , sungguh menye...","kamar téh genah pisan jeung beresih , matak pi...",[A] kamar [O] sangat nyaman [S] positive\n[A] ...,"kamar téh ngeunaheun pisan jeung beresih , mat...",[A] kamar [O] genah pisan [S] positive\n[A] ka...,[A] kamar [O] ngeunaheun pisan [S] positive\n[...,FALSE,NaN
998,4498,"kamarnya luas , kasurnya empuk , kamar mandiny...","kamarna lega , kasurna empuk , kamar mandina o...",[A] kamarnya [O] luas [S] positive\n[A] kasurn...,"kamarna lega , kasurna empuk , kamar mandina o...",[A] kamarna [O] lega [S] positive\n[A] kasurna...,[A] kamarna [O] lega [S] positive\n[A] kasurna...,FALSE,NaN


In [35]:
df_translated_correction.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   sentence_id                     1000 non-null   int64 
 1   input                           1000 non-null   object
 2   input_sunda                     1000 non-null   object
 3   target_format                   1000 non-null   object
 4   input_sunda_corrected           767 non-null    object
 5   target_format_sunda             1000 non-null   object
 6   target_format_sunda_corrected   697 non-null    object
 7   aspect_or_opinion_not_in_input  1000 non-null   object
 8   mismatch_notes_format           34 non-null     object
dtypes: int64(1), object(8)
memory usage: 70.4+ KB


In [36]:
df_translated_correction[f'target_format_{lang_target}'].fillna('[A] [O] [S]', inplace=True)

/tmp/ipykernel_3224141/3226597203.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_translated_correction[f'target_format_{lang_target}'].fillna('[A] [O] [S]', inplace=True)


In [37]:
df_translated_correction[f'input_{lang_target}_corrected'] = df_translated_correction[f'input_{lang_target}_corrected'].apply(lambda x: x.replace('’', "'") if isinstance(x, str) else x)
df_translated_correction[f'target_format_{lang_target}_corrected'] = df_translated_correction[f'target_format_{lang_target}_corrected'].apply(lambda x: x.replace('’', "'") if isinstance(x, str) else x)
df_translated_correction[f'input_{lang_target}'] = df_translated_correction[f'input_{lang_target}'].apply(lambda x: x.replace('’', "'"))
df_translated_correction[f'target_format_{lang_target}'] = df_translated_correction[f'target_format_{lang_target}'].apply(lambda x: x.replace('’', "'"))

In [38]:
list(df_translated_correction.loc[df_translated_correction['sentence_id'] == 433, f'input_{lang_target}_corrected'])

[]

In [39]:
# Strip strings of all _corrected columns
df_translated_correction[f'input_{lang_target}_corrected'] = df_translated_correction[f'input_{lang_target}_corrected'].apply(lambda x: x.strip() if isinstance(x, str) else x)
df_translated_correction[f'target_format_{lang_target}_corrected'] = df_translated_correction[f'target_format_{lang_target}_corrected'].apply(lambda x: x.strip() if isinstance(x, str) else x)

# Set to pandas nan if empty string
df_translated_correction[f'input_{lang_target}_corrected'] = df_translated_correction[f'input_{lang_target}_corrected'].apply(lambda x: x if x != '' else np.nan)
df_translated_correction[f'target_format_{lang_target}_corrected'] = df_translated_correction[f'target_format_{lang_target}_corrected'].apply(lambda x: x if x != '' else np.nan)

In [40]:
# Fill _corrected columns with input_eng if null
df_translated_correction[f'input_{lang_target}_corrected'] = df_translated_correction[f'input_{lang_target}_corrected'].fillna(df_translated_correction[f'input_{lang_target}'])
df_translated_correction[f'target_format_{lang_target}_corrected'] = df_translated_correction[f'target_format_{lang_target}_corrected'].fillna(df_translated_correction[f'target_format_{lang_target}'])

In [41]:
df_translated_correction.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   sentence_id                     1000 non-null   int64 
 1   input                           1000 non-null   object
 2   input_sunda                     1000 non-null   object
 3   target_format                   1000 non-null   object
 4   input_sunda_corrected           1000 non-null   object
 5   target_format_sunda             1000 non-null   object
 6   target_format_sunda_corrected   1000 non-null   object
 7   aspect_or_opinion_not_in_input  1000 non-null   object
 8   mismatch_notes_format           34 non-null     object
dtypes: int64(1), object(8)
memory usage: 70.4+ KB


In [42]:
def check_mismatches_triplet_format_sent_id(outputs_text):
	mismatch_indexes = []
	mismatch_notes = {}
	for key, instance in outputs_text.items():
		translated_text = add_space_around_punctuation(instance['translated_text'].lower()).strip()
		mismatched = False
		for triplet in instance['translated_triplets']:
			aspect_term = add_space_around_punctuation(triplet['aspect_term'].lower()).strip()
			opinion_term = add_space_around_punctuation(triplet['opinion_term'].lower()).strip()
			if aspect_term not in translated_text and aspect_term != 'null':
				print(f"Mismatch in instance sentence_id {instance['sentence_id']} key {key}: aspect_term '{aspect_term}' not found in {translated_text}")
				mismatch_notes[key] = mismatch_notes.get(key, []) + [f"aspect_term '{aspect_term}' not found"]
				mismatched = True
			if opinion_term not in translated_text and opinion_term != 'null':
				print(f"Mismatch in instance sentence_id {instance['sentence_id']} key {key}: opinion_term '{opinion_term}' not found in {translated_text}")
				mismatch_notes[key] = mismatch_notes.get(key, []) + [f"opinion_term '{opinion_term}' not found"]
				mismatched = True
		if mismatched:
			mismatch_indexes.append(key)
	print(f"Total mismatches found: {len(mismatch_indexes)}")
	return mismatch_indexes, mismatch_notes

In [43]:
outputs_text_correction = {}
for idx, row in df_translated_correction.iterrows():
	outputs_text_correction[idx] = {
		'sentence_id': row['sentence_id'],
		'translated_text': row[f'input_{lang_target}_corrected'],
		'translated_triplets': parse_absa_string(row[f'target_format_{lang_target}_corrected'].replace('\n', ' [SSEP] '))
	}
	# Change the keys of translated_triplets from A, O, S to aspect_term, opinion_term, sentiment_polarity
	for triplet in outputs_text_correction[idx]['translated_triplets']:
		triplet['aspect_term'] = triplet.pop('A', '')
		triplet['opinion_term'] = triplet.pop('O', '')
		triplet['sentiment_polarity'] = triplet.pop('S', '')

In [44]:
len(outputs_text_correction)

1000

In [45]:
outputs_text_correction[372]

{'sentence_id': 3872,
 'translated_text': 'kamar rapih , ngeunaheun , palayanan someah . direkomendasikeun pisan atuh . ngan hanjakal pas muka hordéng jandéla , pamandanganna kurang alus . [A] [O] [S]',
 'translated_triplets': [{'aspect_term': 'kamar',
   'opinion_term': 'rapih , ngeunaheun',
   'sentiment_polarity': 'positive'},
  {'aspect_term': 'palayanan',
   'opinion_term': 'someah',
   'sentiment_polarity': 'positive'},
  {'aspect_term': 'pamandanganana',
   'opinion_term': 'kurang alus',
   'sentiment_polarity': 'negative'},
  {'aspect_term': 'null',
   'opinion_term': 'direkomendasikeun pisan',
   'sentiment_polarity': 'positive'}]}

In [46]:
mismatch_indexes, mismatch_notes = check_mismatches_triplet_format_sent_id(outputs_text_correction)

Mismatch in instance sentence_id 3514 key 14: opinion_term 'genah' not found in tempatna ngeunaheun , jauh ti karaméan lalu lintas kandaraan . [a] [o] [s]
Mismatch in instance sentence_id 3552 key 52: opinion_term 'éksélén' not found in kamar nu alus pisan . [a] [o] [s]
Mismatch in instance sentence_id 3564 key 64: aspect_term 'palayananana' not found in palayananna kurang hadé . [a] [o] [s]
Mismatch in instance sentence_id 3573 key 73: opinion_term 'kasep' not found in alus kamarna . [a] [o] [s]
Mismatch in instance sentence_id 3596 key 96: aspect_term 'sandalna' not found in sendalna euweuh . cenah wifi gratis , ngan gening teu dibéré passwordna , anéh . [a] [o] [s]
Mismatch in instance sentence_id 3602 key 102: opinion_term 'genah' not found in ku harga nu kahontal , urang geus meunang fasilitas nu ngeunaheun . [a] [o] [s]
Mismatch in instance sentence_id 3607 key 107: aspect_term 'éksteriorna' not found in hotélna kawas kurang kaurus , boh éksterior atawa interiorna . punten benerk

In [47]:
df_translated_correction['aspect_or_opinion_not_in_input'] = [idx in mismatch_indexes for idx in df_translated_correction.index]
df_translated_correction['mismatch_notes'] = [mismatch_notes.get(idx, []) for idx in df_translated_correction.index]
df_translated_correction['mismatch_notes_format'] = df_translated_correction['mismatch_notes'].apply(lambda x: '\n'.join(x))

In [48]:
df_translated_correction[['aspect_or_opinion_not_in_input', 'mismatch_notes_format']].to_csv(f'temp/translation_output/{lang_target}/{dataset_folder}/translated_dataset_{lang}_{split}_correction_mismatches.csv', index=False)